# TCC — detecção, rastreamento e contagem de braçadas de nadadores

## 1. Configuração

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# ---------- onde estão os arquivos ----------
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/TCC-nadador')
    TRABALHO = Path('/content/trabalho')
else:
    BASE = Path(os.environ.get('TCC_BASE', Path.home() / 'TCC-nadador'))
    TRABALHO = BASE / 'trabalho'

ZIP_PATH = BASE / 'Dataset' / 'dataset nadador Yolo11.zip'
VIDEO_DIR = BASE / 'videos'
OUTPUT_DIR = BASE / 'resultados'

# ---------- treino principal ----------
MODEL_NAME = 'yolo11m.pt'
RUN_NAME = 'yolo11m_640_por_competicao'   # troque junto com MODEL_NAME para treinar outra variante
IMGSZ = 640
EPOCHS = 150
PATIENCE = 30
BATCH = -1                     # -1 = maior lote que cabe na GPU; se faltar memória, use 8
WORKERS = 4 if IN_COLAB else 2
FRACTION = 1.0
FORCE_REBUILD = False
FORCE_RETRAIN = False
MERGE_RELAX_INTO_SWIMMER = False

# ---------- treinos extras, feitos DEPOIS de todo o resto (lista vazia = nenhum) ----------
# cada item: (pesos iniciais, nome do treino, resolução)
TREINOS_EXTRAS = [('yolo11s.pt', 'yolo11s_640_por_competicao', 640), ('yolo11m.pt', 'yolo11m_960_por_competicao', 960),
                  ('yolo11x.pt', 'yolo11x_640_por_competicao', 640), ('yolo11s.pt', 'yolo11s_960_por_competicao', 960),
                  ('yolo11n.pt', 'yolo11n_640_por_competicao', 640)]

# ---------- modelos anteriores, para comparar (ignorados se o arquivo não existir) ----------
MODELOS_ANTERIORES = {
    'yolo11x_piscina (11/11)': OUTPUT_DIR / 'yolo11x_swimmer_zip' / 'weights' / 'best.pt',
    'yolo11x_drone (10/11)': OUTPUT_DIR / 'treino_yolo11x' / 'weights' / 'best.pt',
}
MODELO_PARA_VIDEOS = None      # None = usa o modelo treinado neste notebook nas seções 7 a 9

# ---------- vídeos ----------
PASSO_VIDEO = 10
NADADORES = {                  # nadadores visíveis em cada vídeo (None = não contado)
    'natacao_teste_07.mp4': 1, 'natacao_teste_08.mp4': 1, 'natacao_teste_09.mp4': None,
    'natacao_teste_3_c1.mp4': 1, 'natacao_teste_3_c2.mp4': 1,
    'natacao_teste_4_c1.mp4': 8, 'natacao_teste_4_c3.mp4': 8, 'natacao_teste_6_c1.mp4': 8,
}
VIDEOS_RASTREAMENTO = ['natacao_teste_4_c1.mp4', 'natacao_teste_4_c3.mp4', 'natacao_teste_07.mp4',
                       'natacao_teste_08.mp4', 'natacao_teste_3_c1.mp4', 'natacao_teste_3_c2.mp4']
MAX_QUADROS = None             # None = vídeo inteiro

# ---------- rastreador ----------
DET_CONF = 0.10                # corte de confiança do detector
TRACK_ACTIVATION = 0.30
MATCH_THRESHOLD = 0.90         # maior = mais permissivo
LOST_BUFFER = 90
LACUNA_MAX = 60                # quadros: lacunas até esse tamanho são interpoladas dentro da raia

# ---------- contador de braçadas sem pose ----------
LIMIAR_NAO_AGUA = 40           # distância de cor (Lab) para um pixel ser considerado "não é água"
BANDA_BRACADA = (0.4, 2.4)     # Hz: faixa plausível de braçadas por segundo
HISTERESE = 0.0                # fração do desvio-padrão; 0 = cruza a média
RODAR_PIPELINE_DO_REPOSITORIO = True   # seção 9: roda também o contador antigo, por pose, para comparar

# contagem manual, cada braço conta uma; nos vídeos de 8 raias, da raia mais à ESQUERDA para a direita
BRACADAS_MANUAIS = {
    'natacao_teste_07.mp4': [9], 'natacao_teste_08.mp4': [18],
    'natacao_teste_3_c1.mp4': [34], 'natacao_teste_3_c2.mp4': [34],
    'natacao_teste_4_c1.mp4': [9, 6, 8, 5, 7, 8, 6, 5],
    'natacao_teste_4_c3.mp4': [15, 13, 14, 13, 13, 17, 11, 10],
}

# ---------- modo de ensaio: roda tudo em minutos, só para testar o notebook ----------
SMOKE = os.environ.get('TCC_SMOKE') == '1'
if SMOKE:
    MODEL_NAME, RUN_NAME, EPOCHS, FRACTION, BATCH, WORKERS = 'yolo11n.pt', 'ensaio_smoke_v3', 1, 0.03, 4, 0
    TREINOS_EXTRAS = [('yolo11n.pt', 'ensaio_smoke_v3_extra', 640)]
    PASSO_VIDEO, MAX_QUADROS, VIDEOS_RASTREAMENTO = 200, 150, ['natacao_teste_4_c1.mp4', 'natacao_teste_07.mp4']
    MODELO_PARA_VIDEOS = MODELOS_ANTERIORES['yolo11x_piscina (11/11)']

PREPARED_DIR = TRABALHO / 'swimmers_por_competicao'
WORK_DIR = TRABALHO / 'work_raw'
DATA_YAML = PREPARED_DIR / 'data.yaml'
REPO_DIR = Path(os.environ.get('TCC_REPO', TRABALHO / 'tcc-swimmer-tracking'))
RUN_DIR = OUTPUT_DIR / RUN_NAME
print('Colab:', IN_COLAB, '| BASE:', BASE, '| treino:', RUN_DIR)

## 2. Ambiente

In [ ]:
import subprocess

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics',
                    'supervision>=0.30.0,<0.31.0', 'motmetrics'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

import copy
import csv
import io
import json
import re
import shutil
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import supervision as sv
import torch
import ultralytics
import yaml
from IPython.display import display
from scipy.signal import butter, filtfilt
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('ultralytics', ultralytics.__version__, '| supervision', sv.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO HÁ — o treino em CPU é inviável')

assert ZIP_PATH.exists(), f'Não achei o conjunto de dados em {ZIP_PATH}'
assert VIDEO_DIR.exists(), f'Não achei a pasta de vídeos em {VIDEO_DIR}'
TRABALHO.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS = sorted(p for p in VIDEO_DIR.glob('*.mp4'))
VIDEOS_RASTREAMENTO = [v for v in VIDEOS_RASTREAMENTO if (VIDEO_DIR / v).exists()]
print(len(VIDEOS), 'vídeos:', [v.name for v in VIDEOS])

## 3. Conjunto de dados, dividido por competição

Teste: `Angers19`, `Gwangju`, `Nice`. Validação: `StRaphael18`, `7Collines19`. Treino: as outras nove competições.
A célula confere sozinha o resultado esperado: 2.264 / 81 / 58 imagens.

In [ ]:
# Competições reservadas: nenhum quadro delas entra no treino.
TEST_COMPS = {'Angers19', 'Gwangju', 'Nice'}
VAL_COMPS = {'StRaphael18', '7Collines19'}


def origem(p):
    """Quadro de origem: nome do arquivo sem o hash do Roboflow."""
    return re.sub(r'\.rf\.[a-zA-Z0-9]+$', '', p.stem)


def competicao(p):
    base = re.sub(r'_(jpg|jpeg|png)$', '', origem(p))
    prova = re.sub(r'[-_]\d+$', '', base)
    return prova.split('_')[0]


if DATA_YAML.exists() and not FORCE_REBUILD:
    print("Dataset já está preparado. Pulando etapa de reconstrução.")
else:
    print("Iniciando preparação do dataset...")
    if PREPARED_DIR.exists():
        shutil.rmtree(PREPARED_DIR)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)

    print("Extraindo ZIP...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)

    yaml_files = list(WORK_DIR.rglob('data.yaml'))
    if not yaml_files:
        raise FileNotFoundError("data.yaml não encontrado no ZIP.")
    dataset_root = yaml_files[0].parent
    with open(yaml_files[0], 'r') as f:
        orig_names = yaml.safe_load(f).get('names', [])
    print(f"Classes originais encontradas: {orig_names}")

    class_map = {}
    for i, name in enumerate(orig_names):
        lower_name = name.lower()
        if lower_name in ['bodysurface', 'bodyunder']:
            class_map[i] = 0
        elif lower_name == 'relax':
            class_map[i] = 0 if MERGE_RELAX_INTO_SWIMMER else -1
        elif len(orig_names) == 1:
            class_map[i] = 0
        else:
            print(f"Aviso: classe desconhecida '{name}'. Será ignorada.")
            class_map[i] = -1

    PREPARED_DIR.mkdir(parents=True)
    for s in ['train', 'valid', 'test']:
        (PREPARED_DIR / s / 'images').mkdir(parents=True, exist_ok=True)
        (PREPARED_DIR / s / 'labels').mkdir(parents=True, exist_ok=True)

    ja_usado = set()
    contagem = defaultdict(int)
    caixas = defaultdict(int)
    comps = defaultdict(set)
    for split_orig in ['train', 'valid', 'test']:
        for img in sorted((dataset_root / split_orig / 'images').glob('*.*')):
            comp = competicao(img)
            destino = 'test' if comp in TEST_COMPS else 'valid' if comp in VAL_COMPS else 'train'
            # O treino original tem 9 cópias aumentadas de cada quadro:
            # na validação e no teste entra só uma cópia por quadro.
            if destino != 'train' and split_orig == 'train':
                if origem(img) in ja_usado:
                    continue
                ja_usado.add(origem(img))
            shutil.copy(img, PREPARED_DIR / destino / 'images' / img.name)
            lbl_src = dataset_root / split_orig / 'labels' / f"{img.stem}.txt"
            lbl_dst = PREPARED_DIR / destino / 'labels' / f"{img.stem}.txt"
            linhas = lbl_src.read_text().splitlines() if lbl_src.exists() else []
            with open(lbl_dst, 'w') as f:
                for line in linhas:
                    parts = line.split()
                    if not parts:
                        continue
                    novo = class_map.get(int(parts[0]), -1)
                    if novo != -1:
                        f.write(f"{novo} {' '.join(parts[1:])}\n")
                        caixas[destino] += 1
            contagem[destino] += 1
            comps[destino].add(comp)

    new_yaml = {
        'path': str(PREPARED_DIR.absolute()),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 1,
        'names': ['swimmer'],
    }
    with open(DATA_YAML, 'w') as f:
        yaml.dump(new_yaml, f, sort_keys=False)

    shutil.rmtree(WORK_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print("Imagens por divisão:", dict(contagem))
    print("Caixas de nadador por divisão:", dict(caixas))
    for s in ['train', 'valid', 'test']:
        print(f"  {s}: {sorted(comps[s])}")
    esperado = {'train': 2264, 'valid': 81, 'test': 58}
    assert dict(contagem) == esperado, f"contagem diferente da esperada {esperado}"
    assert not (comps['train'] & comps['valid']) and not (comps['train'] & comps['test'])
    print("✅ Dataset preparado, dividido por competição.")

## 4. Treino

Não treina de novo se já existe treino concluído com o mesmo nome; retoma se o treino parou no meio.

In [ ]:
def treina(model_name, run_name, imgsz):
    """Treina (ou retoma, ou reaproveita) e devolve o caminho do best.pt."""
    run_dir = OUTPUT_DIR / run_name
    last, best = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt'
    estado = 'NOVO'
    if last.exists():
        ckpt = torch.load(last, map_location='cpu', weights_only=False)
        estado = 'CONCLUIDO' if (ckpt.get('epoch', 0) == -1 or not ckpt.get('optimizer')) else 'RETOMAR'
    if estado == 'CONCLUIDO' and FORCE_RETRAIN:
        estado = 'NOVO'
    print(f'{run_name}: {estado}')
    if estado == 'RETOMAR':
        YOLO(str(last)).train(resume=True)
    elif estado == 'NOVO':
        YOLO(model_name).train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=imgsz, batch=BATCH, device=DEVICE, workers=WORKERS,
            project=str(OUTPUT_DIR), name=run_name, exist_ok=True, patience=PATIENCE, cos_lr=True,
            close_mosaic=10, cache='disk', amp=True, plots=True, seed=42, fraction=FRACTION,
        )
    assert best.exists(), f'O best.pt de {run_name} não foi gerado.'
    return best


BEST_MODEL = treina(MODEL_NAME, RUN_NAME, IMGSZ)
print('Melhor modelo:', BEST_MODEL)

## 5. Avaliação em imagens anotadas

O modelo deste notebook nunca viu as competições da validação e do teste. O `yolo11x_piscina (11/11)` viu, então o número dele é otimista.

In [ ]:
def avalia(pesos, split, imgsz=IMGSZ):
    r = YOLO(str(pesos)).val(data=str(DATA_YAML), split=split, imgsz=imgsz, device=DEVICE, plots=False,
                             verbose=False, workers=0, project=str(TRABALHO / 'val_tmp'), name='v', exist_ok=True)
    return {'precisao': round(float(r.box.mp), 3), 'recall': round(float(r.box.mr), 3),
            'mAP50': round(float(r.box.map50), 3), 'mAP50_95': round(float(r.box.map), 3)}


MODELOS = {RUN_NAME: BEST_MODEL}
MODELOS.update({nome: p for nome, p in MODELOS_ANTERIORES.items() if Path(p).exists()})
linhas = []
for nome, pesos in MODELOS.items():
    for split in ['val', 'test']:
        linhas.append({'modelo': nome, 'split': split, **avalia(pesos, split)})
tabela_imagens = pd.DataFrame(linhas)
tabela_imagens.to_csv(RUN_DIR / 'avaliacao_imagens.csv', index=False)
display(tabela_imagens)

## 6. Comparação dos modelos nos vídeos

Caixas por quadro de cada modelo, comparadas ao número real de nadadores. Olhe também os quadros em `quadros_videos/`.

In [ ]:
def detecta(modelo, frame, conf=0.05):
    r = modelo.predict(frame, conf=conf, iou=0.45, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return np.empty((0, 4), dtype=np.float32), np.empty((0,), dtype=np.float32)
    return r.boxes.xyxy.cpu().numpy().astype(np.float32), r.boxes.conf.cpu().numpy().astype(np.float32)


def desenha(frame, caixas, conf, titulo):
    img = frame.copy()
    for (x1, y1, x2, y2), c in zip(caixas, conf):
        if c < 0.10:
            continue
        cor = (0, 255, 0) if c >= 0.25 else (0, 220, 255)
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), cor, 2 if c >= 0.25 else 1)
        cv2.putText(img, f'{c:.2f}', (int(x1), max(10, int(y1) - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, cor, 1, cv2.LINE_AA)
    cv2.rectangle(img, (0, 0), (img.shape[1], 18), (0, 0, 0), -1)
    cv2.putText(img, f'{titulo} | conf>=0.25: {int((conf >= 0.25).sum())}', (4, 13),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return img


pasta_quadros = RUN_DIR / 'quadros_videos'
pasta_quadros.mkdir(parents=True, exist_ok=True)
carregados = {nome: YOLO(str(p)) for nome, p in MODELOS.items()}
linhas = []
for video in VIDEOS:
    cap = cv2.VideoCapture(str(video))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    amostrados = list(range(0, total, PASSO_VIDEO))
    meio = amostrados[len(amostrados) // 2]
    acum = {nome: {'q': 0, 'n25': 0, 'sem': 0, 'soma': 0.0} for nome in carregados}
    f = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % PASSO_VIDEO == 0:
            paineis = []
            for nome, modelo in carregados.items():
                caixas, conf = detecta(modelo, frame)
                bons = conf >= 0.25
                a = acum[nome]
                a['q'] += 1; a['n25'] += int(bons.sum()); a['sem'] += int(not bons.any()); a['soma'] += float(conf[bons].sum())
                if f == meio:
                    paineis.append(desenha(frame, caixas, conf, nome))
            if f == meio:
                cv2.imwrite(str(pasta_quadros / f'{video.stem}_{f:05d}.jpg'), np.hstack(paineis))
        f += 1
    cap.release()
    for nome, a in acum.items():
        linhas.append({'video': video.name, 'nadadores': NADADORES.get(video.name), 'modelo': nome,
                       'quadros': a['q'], 'det_por_quadro': round(a['n25'] / max(a['q'], 1), 2),
                       'quadros_sem_deteccao_pct': round(100 * a['sem'] / max(a['q'], 1), 1),
                       'conf_media': round(a['soma'] / a['n25'], 3) if a['n25'] else 0.0})
del carregados
tabela_videos = pd.DataFrame(linhas)
tabela_videos.to_csv(RUN_DIR / 'comparacao_videos.csv', index=False)
for coluna in ['det_por_quadro', 'quadros_sem_deteccao_pct']:
    t = tabela_videos.pivot(index='video', columns='modelo', values=coluna)
    t.insert(0, 'nadadores', [NADADORES.get(v) for v in t.index])
    print(coluna)
    display(t)

## 7. Rastreamento: uma identidade por raia

Cadeia, sobre as detecções de todos os quadros:

1. remove caixas duplicadas sobre o mesmo nadador;
2. ByteTrack com os limiares corrigidos;
3. corta cada trilha onde ela dá um salto lateral (seria uma troca de nadador);
4. agrupa os pedaços em **N raias** pela posição horizontal, cada nadador fica na sua raia e nunca troca;
5. em cada raia fica uma caixa por quadro, e lacunas curtas são interpoladas.

Em vídeo de **um nadador só** não há raias para separar: vale a regra da trilha principal contínua (função `trilha_unica`).

A coluna `exato_pct` é a fração de quadros com exatamente N nadadores. Sem anotação, é a melhor medida disponível.

In [ ]:
PESOS_VIDEOS = Path(MODELO_PARA_VIDEOS) if MODELO_PARA_VIDEOS else BEST_MODEL
pasta_cache = RUN_DIR / 'cache_deteccoes'
pasta_cache.mkdir(parents=True, exist_ok=True)
modelo = YOLO(str(PESOS_VIDEOS))
INFO_VIDEO = {}
for nome in VIDEOS_RASTREAMENTO:
    cap = cv2.VideoCapture(str(VIDEO_DIR / nome))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    INFO_VIDEO[nome] = {'fps': cap.get(cv2.CAP_PROP_FPS), 'w': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                        'h': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                        'n_frames': total if MAX_QUADROS is None else min(total, MAX_QUADROS)}
    destino = pasta_cache / (Path(nome).stem + '.csv')
    if not destino.exists():
        with open(destino, 'w', newline='', encoding='utf-8') as arq:
            w = csv.writer(arq)
            w.writerow(['frame', 'x1', 'y1', 'x2', 'y2', 'conf'])
            for i in range(INFO_VIDEO[nome]['n_frames']):
                ok, frame = cap.read()
                if not ok:
                    break
                caixas, conf = detecta(modelo, frame)
                for (x1, y1, x2, y2), c in zip(caixas, conf):
                    w.writerow([i, f'{x1:.1f}', f'{y1:.1f}', f'{x2:.1f}', f'{y2:.1f}', f'{c:.4f}'])
        print(nome, '-> cache gravado')
    cap.release()


def carrega_cache(csv_path):
    por_frame = defaultdict(list)
    with open(csv_path, newline='', encoding='utf-8') as arq:
        for l in csv.DictReader(arq):
            por_frame[int(l['frame'])].append([float(l[k]) for k in ('x1', 'y1', 'x2', 'y2', 'conf')])
    return por_frame


def suprime_contidas(por_frame, limiar=0.6):
    """Caixa dentro de caixa sobre o mesmo nadador: fica só a de maior confiança."""
    novo = defaultdict(list)
    for fr, ds in por_frame.items():
        mantidas = []
        for d in sorted(ds, key=lambda x: -x[4]):
            area_d = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
            duplicada = False
            for m in mantidas:
                iw, ih = min(d[2], m[2]) - max(d[0], m[0]), min(d[3], m[3]) - max(d[1], m[1])
                if iw > 0 and ih > 0 and iw * ih / max(min(area_d, (m[2] - m[0]) * (m[3] - m[1])), 1e-6) > limiar:
                    duplicada = True
                    break
            if not duplicada:
                mantidas.append(d)
        novo[fr] = mantidas
    return novo


def bytetrack(por_frame, n_frames, det_conf, ativacao, casamento, buffer):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        bt = sv.ByteTrack(track_activation_threshold=ativacao, lost_track_buffer=buffer,
                          minimum_matching_threshold=casamento, frame_rate=30)
    tracks = defaultdict(dict)
    for fr in range(n_frames):
        linhas_fr = [d for d in por_frame.get(fr, []) if d[4] >= det_conf]
        if linhas_fr:
            arr = np.asarray(linhas_fr, dtype=np.float32)
            det = sv.Detections(xyxy=arr[:, :4], confidence=arr[:, 4], class_id=np.zeros(len(arr), dtype=int))
        else:
            det = sv.Detections.empty()
            det.confidence = np.empty((0,), dtype=np.float32)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            saida = bt.update_with_detections(det)
        if saida.tracker_id is not None:
            for box, tid in zip(saida.xyxy, saida.tracker_id):
                tracks[int(tid)][fr] = box.astype(float)
    return tracks


def centro_x(b):
    return (b[0] + b[2]) / 2


def corta_nos_saltos(tracks, fator=1.5):
    pedacos = []
    for tr in tracks.values():
        frames = sorted(tr)
        atual = {frames[0]: tr[frames[0]]}
        for a, b in zip(frames[:-1], frames[1:]):
            larg = max(tr[a][2] - tr[a][0], 1.0)
            if abs(centro_x(tr[b]) - centro_x(tr[a])) > fator * larg * max(1, b - a) ** 0.5:
                pedacos.append(atual)
                atual = {}
            atual[b] = tr[b]
        pedacos.append(atual)
    return [p for p in pedacos if len(p) >= 3]


def agrupa_em_raias(pedacos, n):
    """Cada pedaço vai para uma das N raias, pela posição horizontal mediana (k-médias em 1 dimensão)."""
    if not pedacos:
        return np.array([], dtype=int), np.zeros(n)
    xs = np.array([np.median([centro_x(b) for b in p.values()]) for p in pedacos])
    pesos = np.array([len(p) for p in pedacos], float)
    ordem = np.argsort(xs)
    acumulado = np.cumsum(pesos[ordem]) / pesos.sum()
    centros = np.array([xs[ordem][min(np.searchsorted(acumulado, (k + 0.5) / n), len(xs) - 1)] for k in range(n)])
    rot = np.zeros(len(xs), dtype=int)
    for _ in range(50):
        rot = np.argmin(np.abs(xs[:, None] - centros[None, :]), axis=1)
        novos = np.array([np.average(xs[rot == k], weights=pesos[rot == k]) if (rot == k).any() else centros[k]
                          for k in range(n)])
        if np.allclose(novos, centros):
            break
        centros = novos
    return rot, centros


def interpola(cad, max_gap):
    frames = sorted(cad)
    cheio = {}
    for a, b in zip(frames[:-1], frames[1:]):
        cheio[a] = cad[a]
        if 1 < b - a <= max_gap:
            for fr in range(a + 1, b):
                t = (fr - a) / (b - a)
                cheio[fr] = cad[a] * (1 - t) + cad[b] * t
    cheio[frames[-1]] = cad[frames[-1]]
    return cheio


def uma_por_raia(tracks, n):
    """Lista de N raias (esquerda -> direita); cada raia é {quadro: caixa}."""
    pedacos = corta_nos_saltos(tracks)
    rot, centros = agrupa_em_raias(pedacos, n)
    raias = []
    for k in np.argsort(centros):
        do_k = sorted([p for p, r in zip(pedacos, rot) if r == k], key=len, reverse=True)
        cad = {}
        for p in do_k:                      # o pedaço mais longo tem prioridade em cada quadro
            for fr, b in p.items():
                cad.setdefault(fr, b)
        raias.append(interpola(cad, LACUNA_MAX) if cad else {})
    return raias


def centro(b):
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2])


def costura(tracks, max_gap=90, fator_dist=2.0):
    """Junta trilhas que não se sobrepõem no tempo e continuam perto de onde a anterior parou."""
    cadeias = []
    for tid in sorted(tracks, key=lambda t: min(tracks[t])):
        tr = tracks[tid]
        ini = min(tr)
        melhor, melhor_d = None, None
        for i, cad in enumerate(cadeias):
            fim = max(cad)
            gap = ini - fim
            if gap <= 0 or gap > max_gap:
                continue
            diag = max(np.hypot(tr[ini][2] - tr[ini][0], tr[ini][3] - tr[ini][1]), 1.0)
            d = float(np.linalg.norm(centro(tr[ini]) - centro(cad[fim])))
            if d <= fator_dist * diag * (1 + 0.1 * gap) and (melhor_d is None or d < melhor_d):
                melhor, melhor_d = i, d
        if melhor is None:
            cadeias.append(dict(tr))
        else:
            cadeias[melhor].update(tr)
    return cadeias


def trilha_unica(tracks, min_lacuna=30, min_cadeia=10, vel_max=0.15):
    """Vídeo de um nadador: a maior cadeia é a principal; as outras só entram onde ela não existe
    por pelo menos `min_lacuna` quadros e se estiverem no caminho do nadador."""
    cads = sorted(costura(tracks), key=len, reverse=True)
    if not cads:
        return {}
    cad = dict(cads[0])
    for outra in cads[1:]:
        if len(outra) < min_cadeia:
            continue
        frames_c = np.array(sorted(cad))

        def livre(f):
            if f in cad:
                return False
            antes, depois = frames_c[frames_c < f], frames_c[frames_c > f]
            return len(antes) == 0 or len(depois) == 0 or depois[0] - antes[-1] >= min_lacuna

        novos = [f for f in sorted(outra) if livre(f)]
        if len(novos) < min_cadeia:
            continue
        coerente = True
        for f in (novos[0], novos[len(novos) // 2], novos[-1]):
            j = int(frames_c[np.argmin(np.abs(frames_c - f))])
            ref = cad[j]
            diag = max(np.hypot(ref[2] - ref[0], ref[3] - ref[1]), 1.0)
            if np.linalg.norm(centro(outra[f]) - centro(ref)) > diag * (1.5 + vel_max * abs(int(f) - j)):
                coerente = False
                break
        if coerente:
            for f in novos:
                cad[f] = outra[f]
    return cad


def exato(conjunto, n_frames, n):
    c = np.zeros(n_frames, dtype=int)
    for tr in conjunto:
        for fr in tr:
            if fr < n_frames:
                c[fr] += 1
    return round(100 * float((c == n).mean()), 1)


RAIAS = {}
linhas = []
for nome in VIDEOS_RASTREAMENTO:
    n, n_frames = NADADORES.get(nome) or 1, INFO_VIDEO[nome]['n_frames']
    por_frame = carrega_cache(pasta_cache / (Path(nome).stem + '.csv'))
    antigo = bytetrack(por_frame, n_frames, 0.25, 0.15, 0.60, 90)
    novo = bytetrack(suprime_contidas(por_frame), n_frames, DET_CONF, TRACK_ACTIVATION, MATCH_THRESHOLD, LOST_BUFFER)
    if n == 1:
        unica = trilha_unica(novo)
        RAIAS[nome] = [interpola(unica, LACUNA_MAX) if unica else {}]
    else:
        RAIAS[nome] = uma_por_raia(novo, n)
    linhas.append({'video': nome, 'nadadores': n, 'quadros': n_frames,
                   'ids_config_antiga': len(antigo), 'exato_pct_antiga': exato(antigo.values(), n_frames, n),
                   'ids_config_corrigida': len(novo), 'exato_pct_corrigida': exato(novo.values(), n_frames, n),
                   'ids_uma_por_raia': sum(1 for r in RAIAS[nome] if r), 'exato_pct_uma_por_raia': exato(RAIAS[nome], n_frames, n)})
tabela_rastreador = pd.DataFrame(linhas)
tabela_rastreador.to_csv(RUN_DIR / 'rastreador.csv', index=False)
display(tabela_rastreador)

## 8. Contagem de braçadas sem pose

Para cada raia e cada quadro, mede a fração de pixels dentro da caixa que **não têm a cor da água** (a cor dominante do quadro). Esse valor sobe e desce a cada braçada. O sinal é filtrado na faixa plausível de ritmo e cada subida acima da média conta uma braçada, a forma de contar usada em Victor et al. (2017) e Woinoski e Bajić (2021).

Onde o nadador fica sem caixa por mais de um segundo, as braçadas desse trecho são **estimadas pelo ritmo** (coluna `estimadas_nas_lacunas`); a coluna `contadas` traz só o que foi observado.

Nos testes de 12/11, com 20 nadadores, o erro médio foi de 1,4 braçada por nadador, contra 9,3 do contador por pose do repositório (este medido em 18). A tabela abaixo refaz essa medida com o modelo deste notebook.

Também grava um vídeo por entrada com a caixa, o número da raia e a contagem acumulada.

In [ ]:
def passa_banda(x, fps, banda):
    b, a = butter(2, [banda[0] / (fps / 2), banda[1] / (fps / 2)], btype='band')
    return filtfilt(b, a, x - np.mean(x))


def preenche(x, max_gap):
    """Interpola lacunas curtas do sinal; devolve o sinal e a máscara do que é válido."""
    ok = ~np.isnan(x)
    idx = np.arange(len(x))
    if ok.sum() < 5:
        return x.copy(), ok
    cheio = np.interp(idx, idx[ok], x[ok])
    valido = ok.copy()
    i = 0
    while i < len(x):
        if not ok[i]:
            j = i
            while j < len(x) and not ok[j]:
                j += 1
            if j - i <= max_gap and i > 0 and j < len(x):
                valido[i:j] = True
            i = j
        else:
            i += 1
    return cheio, valido


def conta_bracadas(x, fps):
    """Devolve os quadros em que cada braçada foi contada."""
    eventos = []
    cheio, valido = preenche(x, int(1.0 * fps))
    i = 0
    while i < len(valido):
        if valido[i]:
            j = i
            while j < len(valido) and valido[j]:
                j += 1
            if j - i >= int(2.0 * fps):
                y = passa_banda(cheio[i:j], fps, BANDA_BRACADA)
                h, alto = HISTERESE * np.std(y), False
                for k, v in enumerate(y):
                    if not alto and v > h:
                        alto = True
                        eventos.append(i + k)
                    elif alto and v < -h:
                        alto = False
            i = j
        else:
            i += 1
    return eventos


def estimadas_nas_lacunas(x, eventos, fps, minimo_s=1.0):
    """Braçadas estimadas nos trechos SEM caixa (>= minimo_s) entre a 1ª e a última aparição:
    ritmo (do intervalo mediano entre braçadas contadas) x duração da lacuna."""
    if len(eventos) < 3:
        return 0.0
    inter = np.diff(eventos)
    ritmo = fps / np.median(inter[inter < 2.5 * np.median(inter)])     # braçadas por segundo
    ok = ~np.isnan(x)
    idx = np.flatnonzero(ok)
    if len(idx) < 2:
        return 0.0
    lacuna_s, i = 0.0, idx[0]
    while i <= idx[-1]:
        if not ok[i]:
            j = i
            while j <= idx[-1] and not ok[j]:
                j += 1
            if (j - i) / fps >= minimo_s:
                lacuna_s += (j - i) / fps
            i = j
        else:
            i += 1
    return float(ritmo * lacuna_s)


def sinal_nao_agua(video_path, raias, n_frames):
    cap = cv2.VideoCapture(str(video_path))
    sinal = np.full((len(raias), n_frames), np.nan)
    for fr in range(n_frames):
        ok, frame = cap.read()
        if not ok:
            break
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB).astype(np.float32)
        agua = np.median(lab.reshape(-1, 3), axis=0)
        H, W = frame.shape[:2]
        for i, cad in enumerate(raias):
            if fr in cad:
                x1, y1, x2, y2 = cad[fr]
                a, b, c, d = max(0, int(x1)), max(0, int(y1)), min(W, int(x2)), min(H, int(y2))
                if c - a >= 2 and d - b >= 2:
                    sinal[i, fr] = float((np.linalg.norm(lab[b:d, a:c] - agua, axis=2) > LIMIAR_NAO_AGUA).mean())
    cap.release()
    return sinal


CORES = [(0, 255, 0), (255, 180, 0), (0, 200, 255), (255, 0, 255), (0, 128, 255), (255, 255, 0), (180, 105, 255), (0, 255, 180)]
EVENTOS = {}
linhas = []
for nome in VIDEOS_RASTREAMENTO:
    info, raias = INFO_VIDEO[nome], RAIAS[nome]
    sinal = sinal_nao_agua(VIDEO_DIR / nome, raias, info['n_frames'])
    EVENTOS[nome] = [conta_bracadas(sinal[i], info['fps']) for i in range(len(raias))]
    manual = BRACADAS_MANUAIS.get(nome, [])
    for i, ev in enumerate(EVENTOS[nome]):
        m = manual[i] if i < len(manual) else None
        est = estimadas_nas_lacunas(sinal[i], ev, info['fps'])
        total = round(len(ev) + est, 1)
        linhas.append({'video': nome, 'raia': i + 1, 'quadros_com_caixa': len(raias[i]),
                       'cobertura_pct': round(100 * len(raias[i]) / info['n_frames'], 1),
                       'contadas': len(ev), 'estimadas_nas_lacunas': round(est, 1), 'bracadas': total,
                       'manual': m, 'erro': round(abs(total - m), 1) if m is not None else None})
    # vídeo anotado
    saida = RUN_DIR / 'videos_anotados'
    saida.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(VIDEO_DIR / nome))
    out = cv2.VideoWriter(str(saida / (Path(nome).stem + '_raias_bracadas.mp4')), cv2.VideoWriter_fourcc(*'mp4v'),
                          info['fps'], (info['w'], info['h']))
    for fr in range(info['n_frames']):
        ok, frame = cap.read()
        if not ok:
            break
        for i, cad in enumerate(raias):
            if fr in cad:
                x1, y1, x2, y2 = (int(v) for v in cad[fr])
                feitas = sum(1 for e in EVENTOS[nome][i] if e <= fr)
                cv2.rectangle(frame, (x1, y1), (x2, y2), CORES[i % 8], 2)
                cv2.putText(frame, f'{i + 1}: {feitas}', (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, CORES[i % 8], 2, cv2.LINE_AA)
        out.write(frame)
    cap.release()
    out.release()

tabela_bracadas = pd.DataFrame(linhas)
tabela_bracadas.to_csv(RUN_DIR / 'bracadas_sem_pose.csv', index=False)
display(tabela_bracadas)
com_manual = tabela_bracadas.dropna(subset=['manual'])
if len(com_manual) and MAX_QUADROS is None:
    e = com_manual['erro'].astype(float)
    print(f"{len(com_manual)} nadadores com contagem manual | erro médio {e.mean():.2f} braçada(s) | "
          f"dentro de ±1: {100 * (e <= 1).mean():.0f}% | dentro de ±2: {100 * (e <= 2).mean():.0f}% | "
          f"soma {com_manual['bracadas'].sum():.0f}/{int(com_manual['manual'].sum())}")
elif MAX_QUADROS is not None:
    print('MAX_QUADROS está limitando os vídeos: a comparação com a contagem manual não vale neste modo.')

## 9. Comparação: pipeline do repositório, com contador por pose

Opcional (`RODAR_PIPELINE_DO_REPOSITORIO`). Roda o código do repositório `tcc-swimmer-tracking` sem alteração, com os limiares do rastreador corrigidos, para ter o "antes" na mesma tabela. `bracadas_pipeline` soma os dois braços de todos os IDs.

In [ ]:
if RODAR_PIPELINE_DO_REPOSITORIO:
    if not (REPO_DIR / 'src').exists():
        try:
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FilipeVelasco/tcc-swimmer-tracking.git',
                            str(REPO_DIR)], check=True)
        except Exception:
            dados = urllib.request.urlopen('https://github.com/FilipeVelasco/tcc-swimmer-tracking/archive/refs/heads/main.zip').read()
            with zipfile.ZipFile(io.BytesIO(dados)) as z:
                z.extractall(TRABALHO)
            (TRABALHO / 'tcc-swimmer-tracking-main').rename(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    import src.pipeline.run_pipeline as rp
    from src.detection.detector import BaseDetector
    from src.utils.config import PipelineConfig

    class DetectorDoCache(BaseDetector):
        def __init__(self, csv_path, det_conf):
            self.por_frame = carrega_cache(csv_path)
            self.det_conf = det_conf

        def detect(self, frame, frame_idx):
            linhas_fr = [d for d in self.por_frame.get(frame_idx, []) if d[4] >= self.det_conf]
            if not linhas_fr:
                d = sv.Detections.empty()
                d.confidence = np.empty((0,), dtype=np.float32)
                return d
            arr = np.asarray(linhas_fr, dtype=np.float32)
            return sv.Detections(xyxy=arr[:, :4], confidence=arr[:, 4], class_id=np.zeros(len(arr), dtype=int))

    base_cfg = yaml.safe_load((REPO_DIR / 'configs' / 'pipeline_real_video_natacao.yaml').read_text(encoding='utf-8'))
    linhas = []
    for nome in VIDEOS_RASTREAMENTO:
        cfg = copy.deepcopy(base_cfg)
        saida = RUN_DIR / 'pipeline_repositorio' / Path(nome).stem
        saida.mkdir(parents=True, exist_ok=True)
        cfg['video']['source'] = str(VIDEO_DIR / nome)
        cfg['detector'].update(weights_path='yolo11n.pt', conf_threshold=DET_CONF, device=str(DEVICE))
        cfg['tracker'].update(track_activation_threshold=TRACK_ACTIVATION, minimum_matching_threshold=MATCH_THRESHOLD,
                              lost_track_buffer=LOST_BUFFER, lane_constraint_enabled=False)
        cfg['pose'].update(weights_path='yolo11n-pose.pt', device=str(DEVICE), inference_interval_frames=1)
        cfg['roi'] = {'enabled': False, 'points': [], 'lanes': []}
        cfg['evaluation']['enabled'] = False
        cfg['output']['dir'] = str(saida)
        cfg_path = saida / 'config_usada.yaml'
        cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
        cache_csv = pasta_cache / (Path(nome).stem + '.csv')
        rp.build_detector = lambda **kw: DetectorDoCache(cache_csv, DET_CONF)
        r = rp.SwimmerTrackingPipeline(PipelineConfig.load(cfg_path)).run(max_frames=MAX_QUADROS)
        bracadas = sum(int(c.get('left_wrist', 0)) + int(c.get('right_wrist', 0)) for c in r['stroke_counts'].values())
        manual = sum(BRACADAS_MANUAIS[nome]) if nome in BRACADAS_MANUAIS else None
        sem_pose = round(float(tabela_bracadas[tabela_bracadas['video'] == nome]['bracadas'].sum()), 1)
        linhas.append({'video': nome, 'ids_finais': r['tracking']['new_logical_identities'],
                       'reassociacoes': r['tracking']['persistent_reassociations'],
                       'bracadas_pipeline_por_pose': bracadas, 'bracadas_sem_pose': sem_pose, 'manual_total': manual})
    tabela_pipeline = pd.DataFrame(linhas)
    tabela_pipeline.to_csv(RUN_DIR / 'pipeline_repositorio.csv', index=False)
    display(tabela_pipeline)
else:
    print('Seção desligada (RODAR_PIPELINE_DO_REPOSITORIO = False).')

## 10. Treinos extras

Treina as variantes listadas em `TREINOS_EXTRAS` e mede cada uma na validação e no teste, ao lado do treino principal. Serve para a tabela de comparação entre tamanhos de modelo do TCC. Fica por último de propósito: se a máquina for desligada no meio, o resto já está pronto.

In [ ]:
linhas = [{'modelo': RUN_NAME, 'imgsz': IMGSZ, 'split': s, **avalia(BEST_MODEL, s)} for s in ['val', 'test']]
for pesos_iniciais, nome_treino, imgsz in TREINOS_EXTRAS:
    melhor = treina(pesos_iniciais, nome_treino, imgsz)
    for s in ['val', 'test']:
        linhas.append({'modelo': nome_treino, 'imgsz': imgsz, 'split': s, **avalia(melhor, s, imgsz)})
tabela_extras = pd.DataFrame(linhas)
tabela_extras.to_csv(RUN_DIR / 'comparacao_treinos.csv', index=False)
display(tabela_extras)

## 11. Onde ficou cada coisa

Tudo em `resultados/<RUN_NAME>/`: `weights/best.pt`, `avaliacao_imagens.csv`, `comparacao_videos.csv`, `rastreador.csv`, `bracadas_sem_pose.csv`, `videos_anotados/`, `pipeline_repositorio.csv` e `comparacao_treinos.csv`.

In [ ]:
for arq in sorted(RUN_DIR.rglob('*')):
    if arq.is_file() and arq.suffix in {'.csv', '.pt', '.mp4'} and 'cache_deteccoes' not in arq.parts:
        print(f'{arq.stat().st_size / 1e6:8.1f} MB  {arq.relative_to(OUTPUT_DIR)}')